# NMNIST — Surrogate Gradient Comparison

Compare **arctan** and **superspike** surrogate gradients on NMNIST.
The surrogate gradient is the key signal used to train SNNs via gradient
descent.  This notebook sweeps learning rates for both and plots the convergence.

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"

import jax
import jax.numpy as jnp
import haiku as hk
import optax
import spyx
import spyx.nn as snn
import jmp
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib notebook

In [ ]:
policy = jmp.get_policy('half')
hk.mixed_precision.set_policy(hk.Linear, policy)
hk.mixed_precision.set_policy(snn.LIF,   policy)
hk.mixed_precision.set_policy(snn.LI,    policy)

### NMNIST Dataloading

In [ ]:
import tonic
from tonic import datasets, transforms
from torch.utils.data import DataLoader

sensor_size  = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)
n_classes   = 10
sample_T    = 64
channels    = 128

frame_transform = transforms.Compose([
    transforms.ToFrame(sensor_size=sensor_size, n_time_bins=sample_T),
    lambda x: np.minimum(x, 1),
    lambda x: np.packbits(x, axis=0)
])

train_ds = tonic.datasets.NMNIST(save_to='./data', transform=frame_transform, train=True)
test_ds  = tonic.datasets.NMNIST(save_to='./data', transform=frame_transform, train=False)

train_dl = iter(DataLoader(train_ds, batch_size=4096,
                          collate_fn=tonic.collation.PadTensors(batch_first=True),
                          drop_last=True, shuffle=False))
test_dl  = iter(DataLoader(test_ds,  batch_size=1024,
                          collate_fn=tonic.collation.PadTensors(batch_first=True),
                          drop_last=True, shuffle=False))

x_train, y_train = next(train_dl)
x_test,  y_test  = next(test_dl)

x_train = jnp.array(x_train, dtype=jnp.uint8)
y_train = jnp.array(y_train, dtype=jnp.uint8)
x_test  = jnp.array(x_test,  dtype=jnp.uint8)
y_test  = jnp.array(y_test,  dtype=jnp.uint8)

n_input = int(np.prod(x_train.shape[2:]))
print(f"Train: {x_train.shape}  Test: {x_test.shape}  n_input={n_input}")

In [ ]:
SURROGATES = {
    'arctan':     spyx.axn.Axon(spyx.axn.arctan()),
    'superspike': spyx.axn.Axon(spyx.axn.superspike()),
}


def build_snn(surrogate):
    def snn_fn(x):
        x = hk.BatchApply(hk.Linear(channels, with_bias=False))(x)
        core = hk.DeepRNN([
            snn.LIF((channels,), activation=surrogate),
            hk.Linear(channels, with_bias=False),
            snn.LIF((channels,), activation=surrogate),
            hk.Linear(n_classes, with_bias=False),
            snn.LI((n_classes,))
        ])
        spikes, V = hk.dynamic_unroll(
            core, x, core.initial_state(x.shape[0]),
            time_major=False, unroll=16
        )
        return spikes, V
    return hk.without_apply_rng(hk.transform(snn_fn))

In [ ]:
def train_snn(SNN, params, lr=2e-4, epochs=100, batch=128, label='SNN'):
    opt = optax.chain(optax.centralize(), optax.lion(learning_rate=lr))
    opt_state = opt.init(params)

    @jax.jit
    def net_eval(w, e, t):
        traces, _ = SNN.apply(w, e)
        return spyx.fn.integral_crossentropy(traces, t, smoothing=0.2)

    sg = jax.value_and_grad(net_eval)

    @jax.jit
    def train_step(state, batch):
        w, os = state
        e, t = batch
        e = jnp.unpackbits(e, axis=1)
        loss, g = sg(w, e, t)
        u, os = opt.update(g, os, w)
        w = optax.apply_updates(w, u)
        return (w, os), loss

    @jax.jit
    def eval_step(w, batch):
        e, t = batch
        e = jnp.unpackbits(e, axis=1)
        traces, _ = SNN.apply(w, e)
        acc, _ = spyx.fn.integral_accuracy(traces, t)
        loss = net_eval(w, e, t)
        return jnp.array([acc, loss])

    n_train = y_train.shape[0]
    indices = jnp.arange(n_train)
    best_acc   = 0.0
    best_weights = None
    rng = hk.next_rng_key()()

    for ep in tqdm(range(epochs), desc=label):
        rng, = jax.random.split(rng)
        perm = jax.random.permutation(rng, indices)

        for i in range(0, n_train - batch + 1, batch):
            bi = perm[i:i+batch]
            (params, opt_state), _ = train_step(
                (params, opt_state),
                (x_train[bi], y_train[bi])
            )

        acc, loss = eval_step(params, (x_test, y_test))
        if float(acc) > best_acc:
            best_acc   = float(acc)
            best_weights = params

    return best_weights, best_acc

### Sweep learning rates

In [ ]:
LRS = [5e-5, 1e-4, 2e-4, 5e-4, 1e-3]
EPOCHS = 80

results = []

for name, surrogate in SURROGATES.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()} surrogate")
    print(f"{'='*50}")

    SNN = build_snn(surrogate)
    key = jax.random.PRNGKey(0)
    params = SNN.init(rng=key, x=x_train[:128])

    for lr in LRS:
        # fresh params for each run
        params_i = SNN.init(rng=key, x=x_train[:128])
        _, acc = train_snn(SNN, params_i, lr=lr, epochs=EPOCHS, label=f"{name} @ {lr}")
        print(f"  lr={lr:.0e}  →  {acc:.2%}")
        results.append(dict(surrogate=name, lr=lr, accuracy=acc))

print("\nDone.")

### Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(8, 5))
for name in SURROGATES:
    sub = df[df.surrogate == name]
    ax.plot(sub.lr, sub.accuracy * 100, 'o-', label=name, lw=2, markersize=6)

ax.set_xlabel('Learning rate')
ax.set_ylabel('Best test accuracy (%)')
ax.set_title('NMNIST — arctan vs superspike surrogate gradient')
ax.set_xscale('log')
ax.legend()
ax.grid(True, which='both')
plt.tight_layout()
plt.show()

print(df.pivot(index='lr', columns='surrogate', values='accuracy').to_string())